# MovieLens データセットに対する協調フィルタリング

### MovieLens データセット
推薦システムの実践例を説明するために、MovieLens データセットを使用します。これは、米国国立科学財団の資金提供を受け、ミネソタ大学コンピュータサイエンス・工学部の研究室である GroupLens によって公開された映画レビューのデータセットです。

データをダウンロードし、BigQuery テーブルとしてロードします。

## 学習目標
1. 協調フィルタリングと行列分解（Matrix Factorization）を理解する
2. BigQuery を活用して行列分解モデルをトレーニングする
3. BQML を使用して評価を予測する

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]

%env PROJECT=$PROJECT

%load_ext google.cloud.bigquery

In [ ]:
%%bash
rm -r bqml_data
mkdir bqml_data
cd bqml_data
curl -O 'http://files.grouplens.org/datasets/movielens/ml-20m.zip'
unzip ml-20m.zip
yes | bq rm -r $PROJECT:movielens
bq --location=US mk --dataset \
    --description 'Movie Recommendations' \
    $PROJECT:movielens
bq --location=US load --source_format=CSV \
    --autodetect movielens.ratings ml-20m/ratings.csv
bq --location=US load  --source_format=CSV \
    --autodetect movielens.movies_raw ml-20m/movies.csv

## データの探索
これで、<a href="https://console.cloud.google.com/bigquery">BigQuery</a> で 2 つのテーブルが利用可能になっているはずです。

協調フィルタリングは、ユーザーに対する商品のおすすめや、商品に対するユーザーのターゲティングを生成する方法を提供します。出発点は、ユーザー ID、アイテム ID、ユーザーが製品に付けた評価の 3 つの列を持つテーブル <b>movielens.ratings</b> です。このテーブルはスパース（疎）である可能性があります（ユーザーがすべての製品を評価する必要はありません）。次に、評価のみに基づいて、類似したユーザーや類似した製品を見つけ、ユーザーがまだ見たことのない製品に付けるであろう評価を決定します。その後、予測評価が最も高い製品をユーザーにおすすめしたり、予測評価が最も高いユーザーを製品のターゲットにしたりできます。

In [ ]:
%%bigquery --project $PROJECT
SELECT *
FROM movielens.ratings
LIMIT 10

簡単な探索的クエリを実行すると、データセットが 138,000 人以上のユーザー、約 27,000 本の映画、2,000 万件強の評価で構成されていることがわかり、データが正常にロードされたことが確認できます。

In [ ]:
%%bigquery --project $PROJECT
SELECT 
    COUNT(DISTINCT userId) numUsers,
    COUNT(DISTINCT movieId) numMovies,
    COUNT(*) totalRatings
FROM movielens.ratings

以下のクエリを使用して最初の数本の映画を調べると、genres 列がフォーマットされた文字列であることがわかります。

In [ ]:
%%bigquery --project $PROJECT
SELECT *
FROM movielens.movies_raw
WHERE movieId < 5

以下のようにジャンルを配列にパースしてテーブルを書き換えることができます。

In [ ]:
%%bigquery --project $PROJECT
CREATE OR REPLACE TABLE movielens.movies AS
    SELECT * REPLACE(SPLIT(genres, "|") AS genres)
    FROM movielens.movies_raw

In [ ]:
%%bigquery --project $PROJECT
SELECT *
FROM movielens.movies
WHERE movieId < 5

## 行列分解（Matrix factorization）
行列分解は、評価行列をユーザー因子（user factors）とアイテム因子（item factors）と呼ばれる 2 つのベクトルに分解することに基づく協調フィルタリング手法です。ユーザー因子は user_id の低次元表現であり、アイテム因子も同様に item_id を表します。

**注: 行列分解（MF）モデルのトレーニングには BigQuery の定額（flat-rate）契約が必要です。そのため、ここでは外部プロジェクトから事前トレーニング済みモデルを取得します。**

BigQuery で定額料金を有効にしている場合は、このクエリを使用して MF モデルをトレーニングできます。
```SQL

CREATE OR REPLACE MODEL movielens.recommender
options(model_type='matrix_factorization',
        user_col='userId', item_col='movieId', rating_col='rating')
AS

SELECT 
userId, movieId, rating
FROM movielens.ratings

```

In [ ]:
%%bigquery --project $PROJECT
SELECT iteration, loss, duration_ms
-- 注: 独自のモデルを使用する場合は cloud-training-demos を削除してください: 
FROM ML.TRAINING_INFO(MODEL `cloud-training-demos.movielens.recommender`)

## おすすめ（推薦）の作成

トレーニング済みモデルを使用して、おすすめを提供できるようになりました。例えば、userId が 903 のユーザーにおすすめする最適なコメディ映画を見つけてみましょう。以下のクエリでは、トレーニングされたレコメンデーションモデルを渡して ML.PREDICT を呼び出し、予測を実行する movieId と userId のセットを提供しています。この場合、userId は 1 人（903）のみですが、ジャンルに「Comedy」が含まれるすべての映画を対象としています。

In [ ]:
%%bigquery --project $PROJECT
SELECT * FROM
ML.PREDICT(MODEL `cloud-training-demos.movielens.recommender`, (
  SELECT 
    movieId, title, 903 AS userId
  FROM movielens.movies, UNNEST(genres) g
  WHERE g = 'Comedy'
))
ORDER BY predicted_rating DESC
LIMIT 5

## すでに評価済みの映画の除外
もちろん、これにはユーザーが過去にすでに視聴して評価した映画も含まれます。それらを除外しましょう。

**TODO 1**: ユーザー 903 に対して、すでに視聴した映画を含まない予測を作成します。

In [ ]:
%%bigquery --project $PROJECT
SELECT * FROM
ML.PREDICT(MODEL `cloud-training-demos.movielens.recommender`, (
  WITH seen AS (
    SELECT ARRAY_AGG(movieId) AS movies 
    FROM movielens.ratings
    WHERE userId = 903
  )
  SELECT 
    movieId, title, 903 AS userId
  FROM movielens.movies, UNNEST(genres) g, seen
  WHERE # TODO: 視聴済みの映画を除外するようにこの WHERE 句を完成させてください。
))
ORDER BY predicted_rating DESC
LIMIT 5

このユーザーの場合、たまたま同じ映画のセットが得られました。予測評価の上位には、ユーザーがすでに視聴した映画は含まれていませんでした。

## カスタマーターゲティング

前のセクションでは、特定のユーザーに対して最も評価が高い映画を特定する方法を確認しました。場合によっては、特定の商品があり、それを気に入ってくれそうな顧客を見つけなければならないことがあります。例えば、評価が 1 件しかない movieId = 96481（American Mullet）のレビューをさらに増やしたいと考え、最も高い評価を付けてくれそうな 5 人のユーザーにクーポンを送りたいとします。 

**TODO 2**: *American Mullet (2001)* を気に入る可能性が高い上位 5 人のユーザーを見つけます。

In [ ]:
%%bigquery --project $PROJECT
SELECT * FROM
ML.PREDICT(MODEL `cloud-training-demos.movielens.recommender`, (
  SELECT 
    96481 AS movieId, 
    (SELECT title FROM movielens.movies WHERE movieId=96481) title,
    userId
  FROM
    # TODO: すべてのユーザーを選択
))
ORDER BY predicted_rating DESC
LIMIT 5

### すべてのユーザーと映画に対するバッチ予測
すべてのユーザーと映画の組み合わせに対して予測を実行したい場合はどうすればよいでしょうか？前のクエリのように個別のユーザーと映画を抽出する代わりに、トレーニング中に出現したすべての movieId と userId に対してバッチ予測を実行する便利な関数が提供されています。ここでは LIMIT を適用しています。そうしないと、すべてのユーザーと映画の予測が返され、ノートブックがクラッシュしてしまいます。

In [ ]:
%%bigquery --project $PROJECT
SELECT *
FROM ML.RECOMMEND(MODEL `cloud-training-demos.movielens.recommender`)
LIMIT 10

上のセクションで見たように、ユーザーが過去にすでに視聴して評価した映画を除外することが可能です。すでに視聴した映画がデフォルトで除外されない理由は、ユーザーが過去に気に入ったレストランを推薦することが完全に期待されるような状況（例えばレストランの推薦を考えてみてください）が存在するためです。

Copyright 2019 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.